# Long SW07/HLT DSGE Profile on Colab

This notebook is dedicated to the larger `Smets_Wouters_2007_HLT` benchmark. It runs the Python/JAX profile against the Julia/MacroModelling reference on the same synthetic SW07 payload.

Use `Runtime -> Change runtime type -> GPU` before running. The full profile is intended to be a multi-hour run; use `PROFILE_MODE = "calibration"` first if you only want to check that the environment compiles.

## What This Runs

- Clones the Python translation repo and the Julia source repo into `/content`.
- Installs a CUDA-enabled JAX wheel before importing JAX.
- Writes a Colab-specific SW07 long-profile TOML with `/content` paths.
- Executes `benchmarks/profile_validation.py <profile.toml>`.
- Reports first-order solution parity, Kalman likelihood/filter parity, switching-order likelihood parity, NumPyro log-density coverage, and bounded sparse-tree SEP inversion coverage.

NUTS is intentionally disabled in this long SW07 profile because the full SW07 sampler path can still hit JAX derivative limitations in the current Schur/doubling first-order solve stack.

In [ ]:
# Repository settings. This branch contains the long-profile runner updates.
REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_DSGE.git"
BRANCH = "codex/colab-jax-gemini-profile"
JULIA_REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_Estimation.jl.git"

# Runtime mode. Use "calibration" first; switch to "long" for the multi-hour run.
PROFILE_MODE = "long"  # "calibration" or "long"
PROFILE_SCALE = 1.0     # multiply long-run repetition counts if you need to tune wall time

# JAX install choice. Use "auto", "cuda13", or "cuda12" for GPU; use "cpu" only as a fallback.
JAX_BACKEND = "auto"
STRICT_GPU = True
FORCE_REINSTALL_JAX = True
UPDATE_EXISTING_CLONE = True

# Keep Colab thread counts conservative; the GPU handles compiled JAX kernels.
THREAD_COUNT = 2
JULIA_VERSION = "1.12.6"


In [ ]:
import os
import re
import sys
import subprocess
import time
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

def run_text(cmd, cwd=None, check=False):
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    text = (proc.stdout or "") + (proc.stderr or "")
    print(text)
    return text

if "jax" in sys.modules:
    raise RuntimeError("JAX is already imported. Restart the Colab runtime before changing JAX wheels.")

nvidia_smi = run_text(["bash", "-lc", "nvidia-smi || true"])
cuda_major = None
match = re.search(r"CUDA Version:\s*([0-9]+)", nvidia_smi)
if match:
    cuda_major = int(match.group(1))
driver_match = re.search(r"Driver Version:\s*([0-9.]+)", nvidia_smi)
driver_version = driver_match.group(1) if driver_match else "unknown"
print("Detected NVIDIA driver:", driver_version, "CUDA major:", cuda_major)

os.environ["JAX_ENABLE_X64"] = "1"
if JAX_BACKEND == "cpu":
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
else:
    os.environ.pop("JAX_PLATFORMS", None)
    os.environ.pop("JAX_PLATFORM_NAME", None)
    print("LD_LIBRARY_PATH before clearing:", os.environ.get("LD_LIBRARY_PATH", "(not set)"))
    removed_ld_library_path = os.environ.pop("LD_LIBRARY_PATH", None)
    if removed_ld_library_path:
        print("Cleared LD_LIBRARY_PATH so JAX uses pip-installed CUDA libraries.")
    print("LD_LIBRARY_PATH after clearing:", os.environ.get("LD_LIBRARY_PATH", "(not set)"))

ROOT = Path("/content/SurrogateNN_DSGE")
if not ROOT.exists():
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)])
elif UPDATE_EXISTING_CLONE:
    run(["git", "fetch", "origin", BRANCH], cwd=ROOT)
    run(["git", "checkout", BRANCH], cwd=ROOT)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=ROOT)
os.chdir(ROOT)
print("Python repo:", ROOT)

if JAX_BACKEND == "auto":
    if cuda_major is None:
        selected_jax_extra = "cuda13" if STRICT_GPU else "cpu"
    elif cuda_major >= 13:
        selected_jax_extra = "cuda13"
    else:
        selected_jax_extra = "cuda12"
else:
    selected_jax_extra = JAX_BACKEND
if selected_jax_extra not in {"cpu", "cuda12", "cuda13"}:
    raise ValueError(f"Unsupported JAX_BACKEND={selected_jax_extra!r}.")
print("Selected JAX install target:", selected_jax_extra)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
if FORCE_REINSTALL_JAX:
    run([
        sys.executable, "-m", "pip", "uninstall", "-y",
        "jax", "jaxlib", "jax-cuda12-pjrt", "jax-cuda12-plugin",
        "jax-cuda13-pjrt", "jax-cuda13-plugin",
    ], check=False)
jax_package = "jax" if selected_jax_extra == "cpu" else f"jax[{selected_jax_extra}]"
run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", jax_package])
run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "sympy>=1.13,<2", "scipy>=1.10,<2", "numpyro>=0.20", "pytest>=8.0,<9",
])
run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "show", "jax", "jaxlib", "numpyro"], check=False)

sys.path.insert(0, str(ROOT / "src"))
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpyro

print("Python:", sys.version)
print("JAX:", jax.__version__)
print("NumPyro:", numpyro.__version__)
print("JAX x64 enabled:", jax.config.read("jax_enable_x64"))
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())
gpu_devices = [device for device in jax.devices() if device.platform in {"gpu", "cuda"}]
if selected_jax_extra != "cpu" and not gpu_devices:
    raise RuntimeError("CUDA JAX was requested, but JAX did not report a GPU device.")
if selected_jax_extra != "cpu":
    probe = (jnp.ones((256, 256), dtype=jnp.float32) @ jnp.ones((256, 256), dtype=jnp.float32)).block_until_ready()
    print("GPU probe result:", float(probe[0, 0]))

julia_dir = Path(f"/content/julia-{JULIA_VERSION}")
if not julia_dir.exists():
    tarball = f"julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
    url = f"https://julialang-s3.julialang.org/bin/linux/x64/1.12/{tarball}"
    run(["bash", "-lc", f"cd /content && wget -q {url} && tar -xzf {tarball}"])
os.environ["PATH"] = f"{julia_dir}/bin:" + os.environ["PATH"]
run(["julia", "--version"])

JULIA_ROOT = Path("/content/SurrogateNN_Estimation.jl")
if not JULIA_ROOT.exists():
    run(["git", "clone", "--depth", "1", JULIA_REPO_URL, str(JULIA_ROOT)])
elif UPDATE_EXISTING_CLONE:
    run(["git", "fetch", "origin"], cwd=JULIA_ROOT)
    run(["git", "checkout", "main"], cwd=JULIA_ROOT)
    run(["git", "reset", "--hard", "origin/main"], cwd=JULIA_ROOT)
print("Julia repo:", JULIA_ROOT)


In [ ]:
def scaled_reps(value: int) -> int:
    return max(0, int(round(value * PROFILE_SCALE)))

if PROFILE_MODE == "calibration":
    profile_name = "sw07_hlt_calibration"
    periods = 48
    reps = dict(
        solve=1,
        kalman_value=2,
        kalman_per_period=1,
        kalman_paths=0,
        kalman_grad=0,
        gate_stats=1,
        switching_fixed=1,
        switching=1,
        numpyro_log_density=1,
        numpyro_switching_log_density=1,
        sep=0,
    )
    sep_eval_periods = 1
elif PROFILE_MODE == "long":
    profile_name = "large_sw07_hlt_switching_order_3h"
    periods = 240
    reps = dict(
        solve=scaled_reps(3),
        kalman_value=scaled_reps(800),
        kalman_per_period=scaled_reps(150),
        kalman_paths=scaled_reps(40),
        kalman_grad=0,
        gate_stats=scaled_reps(120),
        switching_fixed=scaled_reps(800),
        switching=scaled_reps(1800),
        numpyro_log_density=scaled_reps(600),
        numpyro_switching_log_density=scaled_reps(600),
        sep=scaled_reps(4),
    )
    sep_eval_periods = 4
else:
    raise ValueError("PROFILE_MODE must be 'calibration' or 'long'.")

report_path = ROOT / "docs" / f"benchmark_report_{profile_name}_colab.md"
config_path = ROOT / "benchmarks" / f"{profile_name}_colab.toml"
config_path.write_text(f'''
[global]
upstream_repo = "{JULIA_ROOT}"
python_executable = "{sys.executable}"
julia_executable = "julia"
thread_count = {THREAD_COUNT}
gate_quantile = 0.8
jax_platform_name = "auto"
report_path = "{report_path}"

[[case]]
name = "{profile_name}"
model_symbol = "Smets_Wouters_2007_HLT"
model_path = "{JULIA_ROOT}/models/Smets_Wouters_2007_HLT.jl"
observables = ["dy", "dc", "dinve", "labobs", "pinfobs", "dwobs", "robs"]
parameter_subset = ["cprobp", "cindp", "curvp", "crpi", "csigl", "calfa"]
numpyro_parameter_subset = ["cprobp", "cindp", "curvp"]
numpyro_prior_width_scale = 0.005
numpyro_prior_width_floor = 0.0001
numpyro_log_density_reps = {reps['numpyro_log_density']}
numpyro_switching_log_density_reps = {reps['numpyro_switching_log_density']}
numpyro_nuts_warmup = 0
numpyro_nuts_samples = 0
numpyro_nuts_chains = 1
numpyro_target_accept_prob = 0.8
numpyro_seed = 20260712
state_names = ["dy", "dc", "dinve", "labobs", "pinfobs", "dwobs", "robs"]
periods = {periods}
shock_seed = 20260712
shock_scale_by_exo = {{ ea = 1.0, eb = 1.0, eg = 1.0, em = 1.0, epinf = 1.0, eqs = 1.0, ew = 1.0 }}
measurement_error_scale = 1.0e-8
jitter = 1.0e-9
solve_reps = {reps['solve']}
kalman_value_reps = {reps['kalman_value']}
kalman_per_period_reps = {reps['kalman_per_period']}
kalman_paths_reps = {reps['kalman_paths']}
kalman_grad_reps = {reps['kalman_grad']}
gate_stats_reps = {reps['gate_stats']}
switching_fixed_reps = {reps['switching_fixed']}
switching_reps = {reps['switching']}
sep_reps = {reps['sep']}
gate_periods = 1
gate_mode = "soft"
gate_beta_eps = 2.0
gate_beta_y = 2.0
gate_hard_threshold = 0.5
gate_prob_floor = 1.0e-4
gate_prob_ceiling = 0.9999
soft_mixture = "logsumexp"
sep_eval_periods = {sep_eval_periods}
sep_periods = 3
sep_branching_order = 1
sep_nnodes = 3
sep_sparse_tree = true
sep_maxit = 6
sep_tol = 1.0e-5
sep_accept_tol = 1.0e-3
sep_inv_maxit = 4
sep_inv_step_tol = 1.0e-5
sep_inv_resid_tol = 1.0e-5
sep_inv_lambda = 1.0e-4
''')

print("Profile mode:", PROFILE_MODE)
print("Profile scale:", PROFILE_SCALE)
print("Benchmark config:", config_path)
print(config_path.read_text()[:4000])


In [ ]:
if PROFILE_MODE == "long":
    print("Starting the long SW07 profile. This is expected to take hours on some runtimes.")
else:
    print("Starting the calibration SW07 profile.")

start = time.perf_counter()
run([sys.executable, str(ROOT / "benchmarks" / "profile_validation.py"), str(config_path)], cwd=ROOT)
elapsed = time.perf_counter() - start
print(f"Elapsed wall time: {elapsed / 3600.0:.3f} hours ({elapsed:.1f} seconds)")

print("Report path:", report_path)
report_text = report_path.read_text()
print(report_text[:12000])


## Interpreting The Output

Read the parity section before using timing numbers. If first-order solution, Kalman likelihood, filtered states, or switching likelihood parity is weak, the timing comparison is not meaningful as an estimation benchmark.

If Colab disconnects, rerun in `PROFILE_MODE = "calibration"` first, then reduce `PROFILE_SCALE` for the long profile. For example, `PROFILE_SCALE = 0.5` approximately halves the repeated likelihood/filter workload while preserving the same SW07 model and 240-period sample.